# Scoring Komentar + Agregasi Skor Influencer (C1)

Notebook ini disusun per **BLOK**, tiap blok punya beberapa **cell** bernomor (A1, A2, dst) biar gampang nunjuk lokasi kalau ada error atau mau ganti sesuatu.

| Blok | Isi |
|---|---|
| **A** | Setup, config, load `comments.csv`, exclude `drrichardlee` |
| **B** | Bersihin teks komentar (cleaning) |
| **C** | Load model (3class & binary), prediksi label tiap komentar |
| **D** | Hitung S_k per komentar & agregasi jadi skor C1 per influencer |

Kalau nanti ada error/perubahan, kasih tau "block X cell Y" biar aku langsung paham lokasinya.

---
## BLOK A — Setup, Config, Load Data

**Cell A1**: import library + konfigurasi (path, nama model, mapping label, mapping S_k).
**Cell A2**: fungsi cari folder root repo otomatis.
**Cell A3**: load `comments.csv` mentah + buang semua baris dari `drrichardlee`.

In [19]:
# ==== Cell A1: Import & Konfigurasi ====
import re
import gc
from pathlib import Path

import numpy as np
import pandas as pd

# --- Username yang DIKECUALIKAN, ga dilanjut ke tahap ini ---
EXCLUDE_USERNAMES = ['drrichardlee']

# --- Aturan cleaning (sama kaya praprocessing_komentar.ipynb) ---
MIN_WORDS = 2

# --- Model ---
MODEL_NAME = 'indolem/indobertweet-base-uncased'
MAX_LEN = 56           # sama kaya training (comment-only, ga pake caption)
BATCH_SIZE = 64

# --- Mapping label -> index (harus SAMA PERSIS kaya pas training) ---
LABEL2ID = {
    'binary': {'non-negatif': 0, 'negatif': 1},
    '3class': {'negatif': 0, 'netral': 1, 'positif': 2},
}

# --- Mapping label -> S_k (skor sentimen per komentar) ---
# 3class       : Persamaan 2.1 di keputusan_metodologi.md (positif=+1, netral=0, negatif=-1)
# binary varianA: cuma komentar NEGATIF yang "menghukum" skor, non-negatif dianggap netral (0)
# binary varianB: simetris, negatif=-1 non-negatif=+1 (kaya 3class versi 2 kelas)
SK_MAP = {
    '3class': {'negatif': -1, 'netral': 0, 'positif': 1},
    '3class_hanya_positif': {'negatif': 0, 'netral': 0, 'positif': 1},   # <-- BARU
    'binary_varianA': {'negatif': -1, 'non-negatif': 0},
    'binary_varianB': {'negatif': -1, 'non-negatif': 1},
}

print('Config siap. EXCLUDE_USERNAMES =', EXCLUDE_USERNAMES)


Config siap. EXCLUDE_USERNAMES = ['drrichardlee']


In [2]:
# ==== Cell A2: Cari folder root repo ====
def find_repo_root():
    current = Path.cwd()
    for _ in range(6):
        if (current / '01_pengumpulan_data').is_dir() and (current / '03_Indobert').is_dir():
            return current
        if current.parent == current:
            break
        current = current.parent
    return None


REPO_ROOT = find_repo_root()
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Nggak ketemu folder root repo (butuh '01_pengumpulan_data' dan '03_Indobert' "
        "keduanya ada sebagai subfolder). Jalankan dari dalam folder repo, atau pindahin "
        "notebook ini ke root repo."
    )
print('Repo root:', REPO_ROOT)

# --- Path input & output ---
PATH_COMMENTS = REPO_ROOT / '01_pengumpulan_data' / '05_scrap_komentar' / 'hasil_07_comments_01' / 'comments.csv'
CHECKPOINT_3CLASS = REPO_ROOT / '03_Indobert' / 'models' / '3class' / 'final' / 'checkpoints' / 'checkpoint-692'
CHECKPOINT_BINARY = REPO_ROOT / '03_Indobert' / 'models' / 'binary' / 'final' / 'checkpoints' / 'checkpoint-258'
OUT_DIR = REPO_ROOT / '03_Indobert' / 'Data' / 'hasil_scoring'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Comments  :', PATH_COMMENTS)
print('Ckpt 3class:', CHECKPOINT_3CLASS)
print('Ckpt binary:', CHECKPOINT_BINARY)
print('Output dir :', OUT_DIR)


Repo root: c:\Users\nesha\Tugas_akhir\seleksi-influencer
Comments  : c:\Users\nesha\Tugas_akhir\seleksi-influencer\01_pengumpulan_data\05_scrap_komentar\hasil_07_comments_01\comments.csv
Ckpt 3class: c:\Users\nesha\Tugas_akhir\seleksi-influencer\03_Indobert\models\3class\final\checkpoints\checkpoint-692
Ckpt binary: c:\Users\nesha\Tugas_akhir\seleksi-influencer\03_Indobert\models\binary\final\checkpoints\checkpoint-258
Output dir : c:\Users\nesha\Tugas_akhir\seleksi-influencer\03_Indobert\Data\hasil_scoring


In [3]:
# ==== Cell A3: Load comments.csv + exclude drrichardlee ====
if not PATH_COMMENTS.exists():
    raise FileNotFoundError(f'{PATH_COMMENTS} tidak ditemukan.')

df_raw = pd.read_csv(PATH_COMMENTS, dtype=str, keep_default_na=False, encoding='utf-8')

required = {'username', 'video_id', 'commenter_username', 'comment_text'}
missing = required - set(df_raw.columns)
if missing:
    raise ValueError(f'comments.csv kurang kolom: {missing}')

n0 = len(df_raw)
df = df_raw[~df_raw['username'].isin(EXCLUDE_USERNAMES)].reset_index(drop=True)
print(f'Komentar mentah       : {n0:,}')
print(f'Setelah exclude {EXCLUDE_USERNAMES}: {len(df):,}  (dibuang {n0 - len(df):,})')


Komentar mentah       : 24,463
Setelah exclude ['drrichardlee']: 24,111  (dibuang 352)


---
## BLOK B — Bersihin Teks Komentar

**Cell B1**: fungsi cleaning (regex URL/mention, dedup, buang tanpa huruf, buang <2 kata) — persis sama kayak `praprocessing_komentar.ipynb`.
**Cell B2**: jalanin cleaning-nya, liat laporan sebelum/sesudah tiap langkah.

In [4]:
# ==== Cell B1: Fungsi cleaning ====
URL_RE = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
MENTION_RE = re.compile(r"@[A-Za-z0-9_.]+")
SPACE_RE = re.compile(r"\s+")


def bersihkan_komentar(comments_df, min_words=MIN_WORDS):
    """Langkah2 PERSIS kaya praprocessing_komentar.ipynb:
    buang kosong -> hapus URL/mention -> dedup (video+akun+teks sama) -> buang tanpa huruf
    -> buang <min_words kata. Emoji dibiarin, ga ada stemming/stopword/dsb."""
    df = comments_df.copy()
    df['comment_text_raw'] = df['comment_text']
    awal = len(df)

    def catat(langkah, sebelum, sesudah):
        print(f'  {langkah}  ({sebelum:,} -> {sesudah:,}, dibuang {sebelum - sesudah:,})')

    n0 = len(df)
    df = df[df['comment_text'].str.strip() != '']
    catat('1. Komentar kosong', n0, len(df))

    n0 = len(df)
    df = df.assign(comment_text=(
        df['comment_text']
        .str.replace(URL_RE, ' ', regex=True)
        .str.replace(MENTION_RE, ' ', regex=True)
        .str.replace(SPACE_RE, ' ', regex=True)
        .str.strip()
    ))
    df = df[df['comment_text'] != '']
    catat('2. Hapus URL/mention, jadi kosong', n0, len(df))

    n0 = len(df)
    df = df.drop_duplicates(subset=['video_id', 'commenter_username', 'comment_text'], keep='first')
    catat('3. Duplikat (video+akun+teks sama)', n0, len(df))

    n0 = len(df)
    df = df[df['comment_text'].str.contains(r'[A-Za-z]', regex=True)]
    catat('4. Tanpa huruf (emoji/tanda baca saja)', n0, len(df))

    n0 = len(df)
    df = df[df['comment_text'].str.split().str.len() >= min_words]
    catat(f'5. Kurang dari {min_words} kata', n0, len(df))

    df = df.rename(columns={'comment_text': 'comment_text_clean'}).reset_index(drop=True)
    print(f'  TOTAL: {awal:,} mentah -> {len(df):,} bersih')
    return df


print('Fungsi bersihkan_komentar() siap.')


Fungsi bersihkan_komentar() siap.


In [5]:
# ==== Cell B2: Jalanin cleaning ====
print('Pembersihan komentar:')
df = bersihkan_komentar(df)
df.head()


Pembersihan komentar:
  1. Komentar kosong  (24,111 -> 24,111, dibuang 0)
  2. Hapus URL/mention, jadi kosong  (24,111 -> 24,010, dibuang 101)
  3. Duplikat (video+akun+teks sama)  (24,010 -> 23,827, dibuang 183)
  4. Tanpa huruf (emoji/tanda baca saja)  (23,827 -> 23,054, dibuang 773)
  5. Kurang dari 2 kata  (23,054 -> 22,224, dibuang 830)
  TOTAL: 24,111 mentah -> 22,224 bersih


,username,video_id,comment_id,commenter_username,comment_text_clean,like_count,reply_count,timestamp,scraped_at,comment_text_raw
0,aymanalts,7680219303685180679,7680231538293768980,,itu setelahnya gmna? beneran sembuh? ngga yaki...,0,0,,2026-09-20T21:59:13.354414,itu setelahnya gmna? beneran sembuh? ngga yaki...
1,aymanalts,7680219303685180679,7680224784055976724,,nunggu distich doklip sambil gaya kipas angin 😭🙏,0,0,,2026-09-20T21:59:13.354438,nunggu distich doklip sambil gaya kipas angin 😭🙏
2,aymanalts,7680219303685180679,7680221356752356112,,"Dok, kalo pernah jatuh dari tangga dalam keada...",0,0,,2026-09-20T21:59:13.354441,"Dok, kalo pernah jatuh dari tangga dalam keada..."
3,aymanalts,7680219303685180679,7680237230006895367,,"ngeliat vt ini dalam keadaan acl, ALIAS NGILU ...",0,0,,2026-09-20T21:59:13.354443,"ngeliat vt ini dalam keadaan acl, ALIAS NGILU ..."
4,aymanalts,7680219303685180679,7680256705557644052,,"mau cerita, aku penata anestesi di rs. pernah ...",0,0,,2026-09-20T21:59:13.354445,"mau cerita, aku penata anestesi di rs. pernah ..."


---
## BLOK C — Load Model & Prediksi

**Cell C1**: fungsi load model+tokenizer dari checkpoint, dan fungsi prediksi batch.
**Cell C2**: load model **3class**, prediksi semua komentar bersih.
**Cell C3**: load model **binary**, prediksi semua komentar bersih.

Catatan: ini bagian yang WAJIB jalan di mesin dengan GPU + model-nya ada (ga bisa dites di sandbox).

In [6]:
# ==== Cell C1: Fungsi load model & prediksi batch ====
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

print('torch:', torch.__version__, '| CUDA tersedia:', torch.cuda.is_available())


def load_model_dan_tokenizer(checkpoint_path, model_name=MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(str(checkpoint_path))
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    model.eval()
    return model, tokenizer, device


def prediksi_batch(texts, model, tokenizer, device, max_len=MAX_LEN, batch_size=BATCH_SIZE):
    """Prediksi index label (int) buat list teks, batch demi batch biar ga OOM."""
    semua_pred = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            enc = tokenizer(
                list(batch), truncation=True, padding='max_length',
                max_length=max_len, return_tensors='pt',
            ).to(device)
            logits = model(**enc).logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            semua_pred.extend(preds.tolist())
            if (i // batch_size) % 20 == 0:
                print(f'    {i + len(batch):,}/{len(texts):,} komentar diprediksi...')
    return semua_pred


print('Fungsi load_model_dan_tokenizer() & prediksi_batch() siap.')


c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.14.1+cu126 | CUDA tersedia: True
Fungsi load_model_dan_tokenizer() & prediksi_batch() siap.


In [7]:
# ==== Cell C2: Prediksi model 3CLASS ====
id2label_3class = {v: k for k, v in LABEL2ID['3class'].items()}

print('Load model 3class...')
model_3c, tok_3c, device = load_model_dan_tokenizer(CHECKPOINT_3CLASS)
print('  device:', device)

print('Prediksi (3class)...')
pred_idx_3c = prediksi_batch(df['comment_text_clean'].tolist(), model_3c, tok_3c, device)
df['pred_label_3class'] = [id2label_3class[i] for i in pred_idx_3c]

del model_3c
gc.collect()
if device == 'cuda':
    torch.cuda.empty_cache()

print(df['pred_label_3class'].value_counts())


Load model 3class...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 528.10it/s]


  device: cuda
Prediksi (3class)...
    64/22,224 komentar diprediksi...
    1,344/22,224 komentar diprediksi...
    2,624/22,224 komentar diprediksi...
    3,904/22,224 komentar diprediksi...
    5,184/22,224 komentar diprediksi...
    6,464/22,224 komentar diprediksi...
    7,744/22,224 komentar diprediksi...
    9,024/22,224 komentar diprediksi...
    10,304/22,224 komentar diprediksi...
    11,584/22,224 komentar diprediksi...
    12,864/22,224 komentar diprediksi...
    14,144/22,224 komentar diprediksi...
    15,424/22,224 komentar diprediksi...
    16,704/22,224 komentar diprediksi...
    17,984/22,224 komentar diprediksi...
    19,264/22,224 komentar diprediksi...
    20,544/22,224 komentar diprediksi...
    21,824/22,224 komentar diprediksi...
pred_label_3class
netral     18750
positif     2250
negatif     1224
Name: count, dtype: int64


In [8]:
# ==== Cell C3: Prediksi model BINARY ====
id2label_binary = {v: k for k, v in LABEL2ID['binary'].items()}

print('Load model binary...')
model_bin, tok_bin, device = load_model_dan_tokenizer(CHECKPOINT_BINARY)
print('  device:', device)

print('Prediksi (binary)...')
pred_idx_bin = prediksi_batch(df['comment_text_clean'].tolist(), model_bin, tok_bin, device)
df['pred_label_binary'] = [id2label_binary[i] for i in pred_idx_bin]

del model_bin
gc.collect()
if device == 'cuda':
    torch.cuda.empty_cache()

print(df['pred_label_binary'].value_counts())


Load model binary...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 744.99it/s]


  device: cuda
Prediksi (binary)...
    64/22,224 komentar diprediksi...
    1,344/22,224 komentar diprediksi...
    2,624/22,224 komentar diprediksi...
    3,904/22,224 komentar diprediksi...
    5,184/22,224 komentar diprediksi...
    6,464/22,224 komentar diprediksi...
    7,744/22,224 komentar diprediksi...
    9,024/22,224 komentar diprediksi...
    10,304/22,224 komentar diprediksi...
    11,584/22,224 komentar diprediksi...
    12,864/22,224 komentar diprediksi...
    14,144/22,224 komentar diprediksi...
    15,424/22,224 komentar diprediksi...
    16,704/22,224 komentar diprediksi...
    17,984/22,224 komentar diprediksi...
    19,264/22,224 komentar diprediksi...
    20,544/22,224 komentar diprediksi...
    21,824/22,224 komentar diprediksi...
pred_label_binary
non-negatif    21213
negatif         1011
Name: count, dtype: int64


---
## BLOK D — Hitung S_k & Agregasi Skor Influencer (C1)

**Cell D1**: hitung S_k per komentar (3 skema: 3class, binary varian A, binary varian B).
**Cell D2**: simpan hasil level-komentar ke `komentar_discoring.csv`.
**Cell D3**: agregasi jadi C1 per influencer (rata-rata S_k, Persamaan 2.2).
**Cell D4**: simpan `skor_influencer.csv` + preview hasil.

In [ ]:
# ==== Cell D1: Hitung S_k per komentar ====
def hitung_sk(pred_labels, sk_map):
    """pred_labels = array nama label (str) hasil prediksi model. Return array S_k (int)."""
    tidak_dikenal = set(pred_labels) - set(sk_map.keys())
    if tidak_dikenal:
        raise ValueError(f'Ada label prediksi yg ga ada di sk_map: {tidak_dikenal}')
    return np.array([sk_map[lbl] for lbl in pred_labels])


df['sk_3class'] = hitung_sk(df['pred_label_3class'], SK_MAP['3class'])
df['sk_3class_hanya_positif'] = hitung_sk(df['pred_label_3class'], SK_MAP['3class_hanya_positif'])   # <-- BARU
df['sk_binary_varianA'] = hitung_sk(df['pred_label_binary'], SK_MAP['binary_varianA'])
df['sk_binary_varianB'] = hitung_sk(df['pred_label_binary'], SK_MAP['binary_varianB'])

df[['comment_text_clean', 'pred_label_3class', 'sk_3class',
    'pred_label_binary', 'sk_binary_varianA', 'sk_binary_varianB']].head()


,comment_text_clean,pred_label_3class,sk_3class,pred_label_binary,sk_binary_varianA,sk_binary_varianB
0,itu setelahnya gmna? beneran sembuh? ngga yaki...,netral,0,non-negatif,0,1
1,nunggu distich doklip sambil gaya kipas angin 😭🙏,netral,0,non-negatif,0,1
2,"Dok, kalo pernah jatuh dari tangga dalam keada...",netral,0,non-negatif,0,1
3,"ngeliat vt ini dalam keadaan acl, ALIAS NGILU ...",netral,0,non-negatif,0,1
4,"mau cerita, aku penata anestesi di rs. pernah ...",netral,0,non-negatif,0,1


In [10]:
# ==== Cell D2: Simpan level-komentar ====
kolom_simpan = ['username', 'video_id', 'commenter_username', 'comment_text_raw',
                 'comment_text_clean', 'pred_label_3class', 'pred_label_binary',
                 'sk_3class', 'sk_binary_varianA', 'sk_binary_varianB']
df[kolom_simpan].to_csv(OUT_DIR / 'komentar_discoring.csv', index=False, encoding='utf-8')
print(f'Tersimpan: {OUT_DIR / "komentar_discoring.csv"}  ({len(df):,} baris)')


Tersimpan: c:\Users\nesha\Tugas_akhir\seleksi-influencer\03_Indobert\Data\hasil_scoring\komentar_discoring.csv  (22,224 baris)


In [23]:
# ==== Cell D3: Agregasi C1 per influencer ====
def agregasi_c1(df, kolom_sk, kolom_influencer='username'):
    """C1 = rata-rata S_k per influencer (Persamaan 2.2)."""
    agg = df.groupby(kolom_influencer)[kolom_sk].agg(
        C1='mean', P='count', total_Sk='sum',
    ).reset_index()
    return agg

def gabung_c1_semua_skema(df):
    hasil = None
    for nama_skema, kolom_sk in [
        ('3class', 'sk_3class'),
        ('3class_hanya_positif', 'sk_3class_hanya_positif'),   # <-- BARU
        ('binary_varianA', 'sk_binary_varianA'),
        ('binary_varianB', 'sk_binary_varianB'),
    ]:
        agg = agregasi_c1(df, kolom_sk)
        agg = agg.rename(columns={
            'C1': f'C1_{nama_skema}', 'P': f'P_{nama_skema}', 'total_Sk': f'total_Sk_{nama_skema}',
        })
        hasil = agg if hasil is None else hasil.merge(agg, on='username', how='outer')
    return hasil


skor_influencer = gabung_c1_semua_skema(df)
skor_influencer = skor_influencer.sort_values('C1_3class', ascending=False).reset_index(drop=True)
skor_influencer.head()


,username,C1_3class,P_3class,total_Sk_3class,C1_3class_hanya_positif,P_3class_hanya_positif,total_Sk_3class_hanya_positif,C1_binary_varianA,P_binary_varianA,total_Sk_binary_varianA,C1_binary_varianB,P_binary_varianB,total_Sk_binary_varianB
0,dokter.harmonospanak,0.326241,282,92,0.343972,282,97,-0.017730,282,-5,0.964539,282,272
1,reisabrotoasmoro,0.317757,107,34,0.336449,107,36,-0.028037,107,-3,0.943925,107,101
2,nisaobgyn,0.300000,10,3,0.300000,10,3,0.000000,10,0,1.000000,10,10
3,doktermedok,0.184089,1081,199,0.205365,1081,222,-0.018501,1081,-20,0.962997,1081,1041
4,doktermatahendra,0.178161,174,31,0.178161,174,31,0.000000,174,0,1.000000,174,174


In [24]:
# ==== Cell D4: Simpan & preview hasil akhir ====
skor_influencer.to_csv(OUT_DIR / 'skor_influencer.csv', index=False, encoding='utf-8')
print(f'Tersimpan: {OUT_DIR / "skor_influencer.csv"}  ({len(skor_influencer):,} influencer)')
print('\nPreview (urut C1_3class tertinggi):')
skor_influencer


Tersimpan: c:\Users\nesha\Tugas_akhir\seleksi-influencer\03_Indobert\Data\hasil_scoring\skor_influencer.csv  (42 influencer)

Preview (urut C1_3class tertinggi):


,username,C1_3class,P_3class,total_Sk_3class,C1_3class_hanya_positif,P_3class_hanya_positif,total_Sk_3class_hanya_positif,C1_binary_varianA,P_binary_varianA,total_Sk_binary_varianA,C1_binary_varianB,P_binary_varianB,total_Sk_binary_varianB
0,dokter.harmonospanak,0.326241,282,92,0.343972,282,97,-0.017730,282,-5,0.964539,282,272
1,reisabrotoasmoro,0.317757,107,34,0.336449,107,36,-0.028037,107,-3,0.943925,107,101
2,nisaobgyn,0.300000,10,3,0.300000,10,3,0.000000,10,0,1.000000,10,10
3,doktermedok,0.184089,1081,199,0.205365,1081,222,-0.018501,1081,-20,0.962997,1081,1041
4,doktermatahendra,0.178161,174,31,0.178161,174,31,0.000000,174,0,1.000000,174,174
5,giapratamamd,0.168188,1534,258,0.224250,1534,344,-0.047588,1534,-73,0.904824,1534,1388
6,dr.ziee,0.148464,586,87,0.199659,586,117,-0.052901,586,-31,0.894198,586,524
7,gelarsramdhani,0.146667,150,22,0.206667,150,31,-0.093333,150,-14,0.813333,150,122
8,meta.hanindita,0.112903,62,7,0.112903,62,7,-0.032258,62,-2,0.935484,62,58
9,doktervito,0.107143,56,6,0.107143,56,6,0.000000,56,0,1.000000,56,56


---
## BLOK E — Cek Manual Sampel Komentar Negatif per Influencer

Buat validasi: ngecek apa prediksi model emang masuk akal buat influencer yang hasil C1-nya keliatan ga sesuai ekspektasi (misal influencer besar kayak dr Tirta/dr Gia malah skornya rendah). Ini bukan nambah data baru, cuma baca ulang hasil yang udah disimpen terus nampilin beberapa sampel komentar yang diprediksi negatif buat dibaca manual.

**Cell E1**: load ulang `komentar_discoring.csv` yang udah disimpen di Blok D (biar notebook ini bisa dijalanin terpisah, ga harus rerun dari Blok A).
**Cell E2**: fungsi `tampilkan_sampel_negatif()` — ambil sampel random komentar yang diprediksi negatif buat username tertentu.
**Cell E3**: jalanin fungsinya buat username yang mau dicek (ganti sesuai kebutuhan) — baca manual, apa prediksinya beneran kritis/nyinyir atau keliatan salah klasifikasi.

In [25]:
# ==== Cell E1: Load ulang hasil scoring (biar ga perlu rerun dari atas) ====
import pandas as pd

df_cek = pd.read_csv(OUT_DIR / 'komentar_discoring.csv', dtype=str, keep_default_na=False, encoding='utf-8')
print(f'Loaded {len(df_cek):,} komentar dari komentar_discoring.csv')

Loaded 22,224 komentar dari komentar_discoring.csv


In [26]:
# ==== Cell E2: Fungsi nampilin sampel komentar negatif per influencer ====
def tampilkan_sampel_negatif(df, usernames, skema='3class', n_sampel=20, seed=42):
    """skema = '3class' atau 'binary'. Ambil sampel RANDOM (bukan yg pertama doang) biar
    representatif, terus print teksnya buat dicek manual."""
    kolom_pred = f'pred_label_{skema}'
    hasil = {}
    for u in usernames:
        sub = df[(df['username'] == u) & (df[kolom_pred] == 'negatif')]
        n_ambil = min(n_sampel, len(sub))
        sampel = sub.sample(n=n_ambil, random_state=seed) if n_ambil > 0 else sub
        hasil[u] = sampel
        print(f"\n=== {u} -- {len(sub)} komentar negatif ({skema}), nampilin {n_ambil} sampel ===")
        for _, row in sampel.iterrows():
            print(f"  - {row['comment_text_clean']}")
    return hasil


print('Fungsi tampilkan_sampel_negatif() siap.')

Fungsi tampilkan_sampel_negatif() siap.


In [15]:
# ==== Cell E3: Jalanin buat cek dr Tirta & dr Gia (ganti usernames sesuai kebutuhan) ====
_ = tampilkan_sampel_negatif(
    df_cek,
    usernames=['tirtacipeng', 'giapratamamd'],
    skema='3class',
    n_sampel=20,
)


=== tirtacipeng -- 58 komentar negatif (3class), nampilin 20 sampel ===
  - bahaya kalo pake jempol kaki...
  - ora iso dok..malah dadi ribut malah aku dibentak²
  - hiii tak puaseee
  - kenapa ini komennya rata rata tentang bola dah? kan konteksnya ga lagi tentang bola
  - JUJUR DR TIRTA MAKIN GANTENG😳
  - takut banget 😭
  - dok aku gamudeng 😭
  - Jangan galak2 Dok saya takuut
  - udah pernah dibahas di youtube kak, katanya mitos. mandi malam cuma bikin dingin kata doktir🤣
  - ganggu orang
  - Kenapa dilidahku greenfield nih pait ya 😭😭😭 mending ultramilk😭😭
  - Dokter tirta penghambat scrolll😭😭🫶
  - pak dokter...Tirta tugasnya dimana
  - ihh pantes aku dispepsia 🥺🥺🥺
  - abisnya cosan pw bgt dok,
  - gini kak ,kamu mau nyepam gimana pun percuma , dokter Tirta emang entertainment nya di Liverpool, algoritma dia selain jadi dokter juga udah ceket di Liverpool,jadi percuma kak mau ngingetin orang gimana ,jelas sebagian besar komennya kalo udah Liverpool kalah jelas yaa komennya disini, do

In [17]:
# ==== Cell E4: Distribusi jumlah & persentase label (3class & binary), seluruh data ====
total = len(df_cek)

print(f'Total komentar yang discoring: {total:,}\n')

print('-- Distribusi pred_label_3class --')
for label, jumlah in df_cek['pred_label_3class'].value_counts().items():
    pct = 100 * jumlah / total
    print(f'  {label:10s}: {jumlah:6,}  ({pct:5.1f}%)')

print('\n-- Distribusi pred_label_binary --')
for label, jumlah in df_cek['pred_label_binary'].value_counts().items():
    pct = 100 * jumlah / total
    print(f'  {label:14s}: {jumlah:6,}  ({pct:5.1f}%)')

Total komentar yang discoring: 22,224

-- Distribusi pred_label_3class --
  netral    : 18,750  ( 84.4%)
  positif   :  2,250  ( 10.1%)
  negatif   :  1,224  (  5.5%)

-- Distribusi pred_label_binary --
  non-negatif   : 21,213  ( 95.5%)
  negatif       :  1,011  (  4.5%)


In [18]:
# ==== Cell E5: Kuantifikasi -- seberapa kuat emoji 😭 nyeret prediksi ke "negatif" ====
def analisis_emoji_vs_label(df, emoji='😭', kolom_pred='pred_label_3class'):
    df = df.copy()
    df['ada_emoji'] = df['comment_text_clean'].str.contains(emoji, regex=False)

    print(f"Proporsi komentar yang ngandung '{emoji}', per label prediksi ({kolom_pred}):")
    tabel = df.groupby(kolom_pred)['ada_emoji'].agg(jumlah_ada_emoji='sum', total='count')
    tabel['persen'] = 100 * tabel['jumlah_ada_emoji'] / tabel['total']
    print(tabel.to_string())

    total_emoji = df['ada_emoji'].sum()
    negatif_dgn_emoji = ((df[kolom_pred] == 'negatif') & df['ada_emoji']).sum()
    total_negatif = (df[kolom_pred] == 'negatif').sum()
    print(f"\nDari {total_negatif} komentar berlabel negatif, {negatif_dgn_emoji} "
          f"({100*negatif_dgn_emoji/total_negatif:.1f}%) ngandung '{emoji}'.")
    print(f"Dari {total_emoji} komentar yang ngandung '{emoji}' total, "
          f"{negatif_dgn_emoji} ({100*negatif_dgn_emoji/total_emoji:.1f}%) dilabel negatif.")
    return tabel


_ = analisis_emoji_vs_label(df_cek, emoji='😭', kolom_pred='pred_label_3class')

Proporsi komentar yang ngandung '😭', per label prediksi (pred_label_3class):
                   jumlah_ada_emoji  total     persen
pred_label_3class                                    
negatif                         300   1224  24.509804
netral                         2440  18750  13.013333
positif                         176   2250   7.822222

Dari 1224 komentar berlabel negatif, 300 (24.5%) ngandung '😭'.
Dari 2916 komentar yang ngandung '😭' total, 300 (10.3%) dilabel negatif.


## BLOK F

In [27]:
# ==== Cell F1: Bandingin C1 dari semua skenario sekaligus ====
def bandingkan_semua_c1(skor_df, kolom_c1=None):
    """Tabel sisi-berdampingan + korelasi rank (Spearman) antar skenario S_k -- buat liat
    seberapa beda rangking influencer kalau skema skoringnya diganti."""
    if kolom_c1 is None:
        kolom_c1 = [c for c in skor_df.columns if c.startswith('C1_')]

    tabel = skor_df[['username'] + kolom_c1].sort_values(kolom_c1[0], ascending=False).reset_index(drop=True)
    print(f'Tabel perbandingan C1 per skenario (urut berdasarkan {kolom_c1[0]}):')
    print(tabel.to_string())

    print('\nKorelasi rank (Spearman) antar skenario -- makin deket 1.0 berarti rangkingnya makin mirip:')
    corr = skor_df[kolom_c1].corr(method='spearman')
    print(corr.round(3).to_string())
    return tabel, corr


tabel_c1, korelasi_c1 = bandingkan_semua_c1(skor_influencer)

Tabel perbandingan C1 per skenario (urut berdasarkan C1_3class):
                 username  C1_3class  C1_3class_hanya_positif  C1_binary_varianA  C1_binary_varianB
0    dokter.harmonospanak   0.326241                 0.343972          -0.017730           0.964539
1        reisabrotoasmoro   0.317757                 0.336449          -0.028037           0.943925
2               nisaobgyn   0.300000                 0.300000           0.000000           1.000000
3             doktermedok   0.184089                 0.205365          -0.018501           0.962997
4        doktermatahendra   0.178161                 0.178161           0.000000           1.000000
5            giapratamamd   0.168188                 0.224250          -0.047588           0.904824
6                 dr.ziee   0.148464                 0.199659          -0.052901           0.894198
7          gelarsramdhani   0.146667                 0.206667          -0.093333           0.813333
8          meta.hanindita   0.11290

In [29]:
# ==== Cell F2: Rekap jumlah negatif/netral/positif per dokter + sampel komentar positif ====
def rekap_label_per_influencer(df, kolom_pred='pred_label_3class', usernames=None):
    """Tabel jumlah & persen komentar per label (negatif/netral/positif), per influencer."""
    sub = df if usernames is None else df[df['username'].isin(usernames)]
    rekap = sub.groupby(['username', kolom_pred]).size().unstack(fill_value=0)
    rekap['total'] = rekap.sum(axis=1)
    for kolom in rekap.columns:
        if kolom != 'total':
            rekap[f'{kolom}_pct'] = (100 * rekap[kolom] / rekap['total']).round(1)
    return rekap


def tampilkan_sampel_label(df, usernames, label, kolom_pred='pred_label_3class', n_sampel=20, seed=42):
    """Generalisasi dari tampilkan_sampel_negatif -- bisa buat label apa aja (positif/netral/negatif)."""
    hasil = {}
    for u in usernames:
        sub = df[(df['username'] == u) & (df[kolom_pred] == label)]
        n_ambil = min(n_sampel, len(sub))
        sampel = sub.sample(n=n_ambil, random_state=seed) if n_ambil > 0 else sub
        hasil[u] = sampel
        print(f"\n=== {u} -- {len(sub)} komentar '{label}' ({kolom_pred}), nampilin {n_ambil} sampel ===")
        for _, row in sampel.iterrows():
            print(f"  - {row['comment_text_clean']}")
    return hasil


# ==== Cell F2 (update): Rekap SEMUA dokter, sampel komentar positif cuma buat 1 dokter ====
print('Rekap jumlah & persen per label (3class) -- SEMUA dokter:')
rekap = rekap_label_per_influencer(df_cek, usernames=None)   # None = semua influencer
print(rekap.to_string())

_ = tampilkan_sampel_label(df_cek, ['dokter.harmonospanak'], label='positif', n_sampel=20)

Rekap jumlah & persen per label (3class) -- SEMUA dokter:
pred_label_3class      negatif  netral  positif  total  negatif_pct  netral_pct  positif_pct
username                                                                                    
aliphildan                 137     915      148   1200         11.4        76.2         12.3
aymanalts                  106     918       72   1096          9.7        83.8          6.6
bidancynthia                10     947       21    978          1.0        96.8          2.1
dionharyadi                114     982       64   1160          9.8        84.7          5.5
dokter.harmonospanak         5     180       97    282          1.8        63.8         34.4
dokter.reni.anak             1     105        6    112          0.9        93.8          5.4
dokter_ika                 101    1360       77   1538          6.6        88.4          5.0
dokteramiraobgyn            47    1225       89   1361          3.5        90.0          6.5
doktermatahe